# v23 Final Run — v22 Baseline + Blur-Robustness Rebuild

Trains **v23**: v22's proven strong baseline (unconstrained head, two-stage training, physics features, signed-loss curriculum, K=4) plus the fixes this project's diagnosis pointed to: sigma fed into the head via a sinusoidal embedding, blur exposure added to BOTH training stages, the blur-deconvolved normalizer and blur-adaptive graph, a zero-init residual-on-PCA structural fallback (so v23 starts training mathematically identical to plain PCA and degrades gracefully instead of arbitrarily under blur), and a calibration mechanism with no post-hoc fitting (a kappa-regularizer plus calibration-aware checkpoint selection).

Full spec: `reports/v23_architecture_plan.tex`. Full build documentation: `smearing_resolution/architecture_experiments/v23_final/README.md`.

**Important, honest limitation (read before you start a long run): `train.py` checkpoints periodically to Drive, but has no auto-resume-from-checkpoint flag** (unlike the earlier `a100_final` run). If Colab disconnects mid-stage, the safest, best checkpoint reached so far is NOT lost (it's already on Drive) — but that stage would need to be restarted from scratch, not resumed exactly where it left off. Stage 1 and Stage 2 are run as **separate cells** specifically so a disconnect during Stage 2 doesn't require re-running Stage 1.

**Before you start — data.** If you already ran the `a100_final` notebook, the two data files are already on your Drive at `MyDrive/siimpl_rot/` and you don't need to re-upload:
- `siimpl_train_v2.csv` (~7.9 GB)
- `siimpl_eval_v2.csv` (~0.9 GB)

Run the cells top to bottom in order.

In [ ]:
# 1. Confirm A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone the code (branch v23-final-build)
#    If the repo is private, set a Colab Secret named GH_TOKEN (key icon in the left sidebar).
import os
GH_TOKEN = os.environ.get('GH_TOKEN', '')
try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or userdata.get('GH_TOKEN')
except Exception:
    pass
REPO = 'github.com/cbharathulwar/sbi-srim.git'
BRANCH = 'v23-final-build'
url = f"https://{(GH_TOKEN + '@') if GH_TOKEN else ''}{REPO}"
%cd /content
!rm -rf sbi-srim
!git clone --branch $BRANCH --single-branch $url sbi-srim
%cd /content/sbi-srim
!git log --oneline -1
!pip -q install scipy 2>/dev/null; echo deps-ok

## 3. Mount Drive, stage data locally, point results at Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

DATA_SRC = '/content/drive/MyDrive/siimpl_rot'
DATA_DST = '/content/sbi-srim/data/siimpl_rot'
os.makedirs(DATA_DST, exist_ok=True)
for f in ['siimpl_train_v2.csv', 'siimpl_eval_v2.csv']:
    s = os.path.join(DATA_SRC, f)
    assert os.path.exists(s), (
        f'MISSING in Drive: {s}\n'
        f'-> Upload it to Google Drive at MyDrive/siimpl_rot/{f} first, then re-run this cell.'
    )
    d = os.path.join(DATA_DST, f)
    print(f'copying {f} ({os.path.getsize(s)/1e9:.2f} GB) ...')
    shutil.copy(s, d)
print('data staged.')
!ls -la /content/sbi-srim/data/siimpl_rot/

# RESULTS live DIRECTLY on Drive -- train.py writes checkpoints periodically
# here (no auto-resume, see the top note -- but nothing achieved so far is lost).
RESULTS_DIR = '/content/drive/MyDrive/v23_final_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results will autosave to:', RESULTS_DIR)

## 4. Estimate training time (measured on THIS GPU, not guessed)

v23's training length is controlled by `MAX_EPOCHS`/`STAGE2_EPOCHS` with early-stopping patience, not a fixed step target -- so this cell runs a short, real, throwaway measurement of actual per-epoch wall-clock time on the real GPU, and reports the WORST-CASE (no-early-stopping) time for both stages, plus a suggested `TIME_BUDGET_HOURS` safety cap. Early stopping will likely finish sooner than the worst case; this gives you a number to plan around either way.

In [ ]:
import os, re, subprocess, time

# train.py logs one line per EPOCH (not per step), formatted like:
#   "  S1 ep   3/300 | train nll=... | val nll=... | lr=... | 140s (loader-wait 5%)"
# Both stages use the SAME SUB_EPOCH_STEPS=1250 decoupling (confirmed in
# train.py's shared run_stage()), so a reduced MAX_TRAIN_TRACKS pool does NOT
# shorten an epoch's step count or its per-step compute cost -- only the
# track-sampling diversity -- so this calibration is a fair measurement of
# real per-epoch wall-clock time on the full-size run.
_EPOCH_RE = re.compile(r'S(\d) ep\s*(\d+)/\d+ \|.*?\|\s*(\d+)s')

def calibrate_epoch_seconds(calib_tracks=20000, calib_seconds_cap=600):
    """Runs train.py on a small track subset for a short wall-clock cap,
    parses real per-epoch seconds for each stage from its own progress lines."""
    calib_dir = '/content/sbi-srim/_calib_v23'
    env = os.environ.copy()
    env.update({
        'KMP_DUPLICATE_LIB_OK': 'TRUE', 'PYTHONIOENCODING': 'utf-8',
        'RESULTS_DIR': calib_dir,
        'TRAIN_CSV': 'data/siimpl_rot/siimpl_train_v2.csv',
        'EVAL_CSV': 'data/siimpl_rot/siimpl_eval_v2.csv',
        'MAX_TRAIN_TRACKS': str(calib_tracks),
        'MAX_EPOCHS': '3', 'STAGE2_EPOCHS': '3', 'PATIENCE': '100', 'STAGE2_PATIENCE': '100',
        'TIME_BUDGET_HOURS': str(calib_seconds_cap / 3600.0),
        'LAMBDA_KAPPA': '0.0',
    })
    print(f'[CALIBRATE] running a short real pass ({calib_tracks} tracks, '
          f'{calib_seconds_cap}s cap) on the real GPU -- a few real epochs of each stage ...')
    result = subprocess.run(
        ['python', '-m', 'smearing_resolution.architecture_experiments.v23_final.train'],
        cwd='/content/sbi-srim', env=env, capture_output=True, text=True)
    out = result.stdout + result.stderr
    matches = _EPOCH_RE.findall(out)
    if not matches:
        print(out[-4000:])
        raise RuntimeError('Could not find any epoch progress lines -- see output above.')
    sec_by_stage = {1: [], 2: []}
    for stage, epoch, sec in matches:
        sec_by_stage[int(stage)].append(int(sec))
    s1_secs = sec_by_stage[1][1:] or sec_by_stage[1]  # drop epoch 1 (cache-warm) if we have more
    s2_secs = sec_by_stage[2][1:] or sec_by_stage[2]
    s1_epoch_sec = sum(s1_secs) / len(s1_secs)
    s2_epoch_sec = sum(s2_secs) / len(s2_secs) if s2_secs else s1_epoch_sec
    print(f'[CALIBRATE] Stage 1: {s1_epoch_sec:.1f}s/epoch (over {len(s1_secs)} measured epochs)')
    print(f'[CALIBRATE] Stage 2: {s2_epoch_sec:.1f}s/epoch (over {len(s2_secs)} measured epochs)')
    return s1_epoch_sec, s2_epoch_sec

S1_EPOCH_SEC, S2_EPOCH_SEC = calibrate_epoch_seconds()

MAX_EPOCHS = 300
STAGE2_EPOCHS = 80
worst_case_hours = (S1_EPOCH_SEC * MAX_EPOCHS + S2_EPOCH_SEC * STAGE2_EPOCHS) / 3600

print(f'\n[ESTIMATE] worst case (no early stopping): '
      f'Stage1 {S1_EPOCH_SEC*MAX_EPOCHS/3600:.1f}h + Stage2 {S2_EPOCH_SEC*STAGE2_EPOCHS/3600:.1f}h '
      f'= ~{worst_case_hours:.1f}h on this GPU')
print('Early-stopping patience (40 epochs Stage 1, 20 epochs Stage 2) will likely '
      'finish sooner than this -- this is a safety-planning upper bound, not a prediction.')

TIME_BUDGET_HOURS = min(worst_case_hours, 24.0)  # sane Colab-session-length cap either way
print(f'\nSuggested TIME_BUDGET_HOURS for the real run: {TIME_BUDGET_HOURS:.1f}h '
      f'(edit this manually below if you want a different cap).')

## 5. Train -- Stage 1 + Stage 2 (single script, both stages)

`train.py` runs Stage 1 (joint training, blur-exposed) then Stage 2 (frozen backbone, fresh head, also blur-exposed) in one process, checkpointing both stages' best weights to Drive as it goes. `LAMBDA_KAPPA=auto` implements the spec's own calibration-regularizer tuning rule on real data at the start of the run.

**No auto-resume exists (see the top note).** If this disconnects, check which checkpoints exist in `RESULTS_DIR` on Drive (cell below) before deciding whether to re-run from the top of Stage 1 or restart just Stage 2 against the saved Stage 1 checkpoint.

In [ ]:
import os
os.environ.update({
    'KMP_DUPLICATE_LIB_OK': 'TRUE',
    'PYTHONIOENCODING': 'utf-8',
    'TRAIN_CSV': 'data/siimpl_rot/siimpl_train_v2.csv',
    'EVAL_CSV': 'data/siimpl_rot/siimpl_eval_v2.csv',
    'RESULTS_DIR': RESULTS_DIR,
    'LAMBDA_KAPPA': 'auto',
    'TIME_BUDGET_HOURS': str(TIME_BUDGET_HOURS),
})
%cd /content/sbi-srim
!python -m smearing_resolution.architecture_experiments.v23_final.train

## 6. Check what's actually on Drive (useful any time, especially after a disconnect)

In [ ]:
!ls -la $RESULTS_DIR

## 7. Evaluate the trained checkpoint

Runs the full 9-sigma x 3-energy-tier grid against the PCA baseline and the real v22 `ROT_FINAL` reference curve (bundled with the code at `v23_final/v22_rot_final_reference_sweep.csv`), printing the pre-registered success/guard cells' pass/fail status directly, plus per-cell mean kappa and top-mixture-weight (the calibration diagnostic).

In [ ]:
import os
os.environ.update({
    'KMP_DUPLICATE_LIB_OK': 'TRUE',
    'PYTHONIOENCODING': 'utf-8',
    'EVAL_CSV': 'data/siimpl_rot/siimpl_eval_v2.csv',
    'CKPT': f'{RESULTS_DIR}/best_checkpoint_stage2.pt',
    'REF_SWEEP_CSV': 'smearing_resolution/architecture_experiments/v23_final/v22_rot_final_reference_sweep.csv',
    'OUT_CSV': f'{RESULTS_DIR}/v23_full_sweep_9tier.csv',
})
%cd /content/sbi-srim
!python -m smearing_resolution.architecture_experiments.v23_final.eval
print(f"\nOutput CSV: {os.environ['OUT_CSV']}  (on Drive)")

## Notes / troubleshooting

- **`best_checkpoint_stage1.pt`, `best_checkpoint_stage2_nll.pt`, `best_checkpoint_stage2.pt`** all land in `RESULTS_DIR` on Drive. `best_checkpoint_stage2.pt` is the calibration-aware-selected one (S{calibration} item 2) -- the one `eval.py` uses by default, and the one that should be reported as v23's result.
- **No auto-resume.** If Stage 1 disconnects partway, re-run cell 5 from scratch (Stage 1 restarts; this is a real limitation of this build, documented honestly rather than assumed away). If Stage 2 disconnects, check whether `best_checkpoint_stage1.pt` exists on Drive first -- if so, a resume-from-frozen-Stage-1 entry point may need to be added; it wasn't part of this build. Ask before assuming one exists.
- **`LAMBDA_KAPPA=auto`** measures the calibration regularizer's weight on real data at the start of the run and prints the value used -- record it, since the spec calls for reporting pre/post-regularizer coverage curves.
- Full architecture documentation, the residual-fallback math, all judgment calls, and validation results are in `smearing_resolution/architecture_experiments/v23_final/README.md` in the cloned repo.